# Yilgarn Au inventory and cohort expansion
Read-only exploration of locally generated results. Run `PYTHONPATH=src uv run python -m gswa_hackathon.expand_yilgarn` from the repository root for the inventory/map, or add `--download` to acquire and audit candidates and build the expanded cohort.

The inventory includes every local WAMEX Yilgarn collar with finite nonnegative Au and valid intervals, not only positive examples or the original 22 holes. Assay counts precede spectral interval matching and are not guaranteed training samples. Negative/censored values are excluded without substitution. No models are fitted in this notebook. Keep all execution outputs local.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "ml_pipeline" else Path.cwd()
RESULT_DIR = ROOT / "data" / "extracted_data" / "yilgarn"
inventory = pd.read_csv(RESULT_DIR / "hole_inventory.csv")
sources = pd.read_csv(RESULT_DIR / "candidate_sources.csv")

print(f"Measured-Au collars: {len(inventory)}")
print(f"Source assay records: {inventory['measured_records'].sum():,}")
print(f"Source records >= 1 ppm: {inventory['anomalies_ge_1ppm'].sum():,}")
print(f"Mineral-log linked collars: {inventory['has_mineral_log'].sum()}")
print(f"Unambiguous sources: {len(sources)}")
print(inventory["mapping_status"].value_counts().to_string())

## Collar map
GDA2020 decimal degrees (EPSG:7844), without a background map or inferred craton boundary. All measured-Au collars are coloured by median Au, maximum Au, and the fraction of source assay records >= 1 ppm. Median and maximum use log1p for display only. Rings identify mineral-log links; red stars identify the pilot. These are collar positions, not the positions of downhole assay intervals. Maxima can be dominated by single outliers; sampling density and interval lengths differ.

In [ ]:
figure, axis = plt.subplots(figsize=(18, 8), layout="constrained")
axis.imshow(plt.imread(RESULT_DIR / "yilgarn_gold_map.png"))
axis.axis("off")
plt.show()
plt.close(figure)

In [ ]:
project_inventory = (
    inventory.assign(positive_hole=inventory["anomalies_ge_1ppm"] > 0)
    .groupby("Project")
    .agg(
        holes=("CollarId", "size"),
        measured_records=("measured_records", "sum"),
        positive_holes=("positive_hole", "sum"),
        source_anomalies=("anomalies_ge_1ppm", "sum"),
        maximum_au_ppm=("max_au_ppm", "max"),
    )
    .sort_values("holes", ascending=False)
)
print(project_inventory.to_string())
print("\nUnknown is a missing-project bucket, not one geological project.")

## Acquisition and matching audit
Run the CLI with `--download` first. Every inventory collar receives an outcome: absent/ambiguous source mapping, download failure, incompatible schema, invalid log, no retained intervals, or ready. Download failures do not mean a hole is unmineralised. Only holes with all ten required descriptor columns can enter the common comparison; partial numeric missingness remains NaN. The original depth tolerance, censoring and conflict rules are unchanged.

Inspect which projects and positive assays survive matching before training. The expanded dataset is written separately; it never replaces `ml_dataset.csv`. Any future split on the expanded cohort is a new experiment, not independent confirmation of the original test result.

In [ ]:
audit = pd.read_csv(RESULT_DIR / "hole_audit.csv")
project_summary = pd.read_csv(RESULT_DIR / "project_summary.csv")
print(audit["status"].value_counts().to_string())
print("\nSource versus model support by project:")
print(project_summary.to_string(index=False))
print("\nExpanded dataset:", RESULT_DIR / "ml_dataset_expanded.csv")